# TrendAI Pipeline End-to-End Execution

Notebook ini mendemonstrasikan eksekusi seluruh pipeline TrendAI secara lengkap, mulai dari:
1. **Scraping** berita retail/FMCG secara real-time dari Google News RSS.
2. **Preprocessing** (pembersihan teks, tokenisasi, eliminasi stopwords).
3. **Topic Modeling & Clustering** (menggunakan TF-IDF + K-Means) ke kategori yang relevan.
4. **AI Summarization** (Extractive menggunakan TextRank lokal, dan Generative menggunakan Groq API secara online).
5. **ROUGE Evaluation** terhadap ground truth buatan manusia yang terstruktur.

In [14]:
import os
import json
import pandas as pd
import logging
from dotenv import load_dotenv

# Load environment variables from .env containing API keys
load_dotenv()

# Setup basic logging to monitor pipeline progression
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
print("Environment and packages loaded successfully.")
print("API Key configured:", "Yes" if os.getenv("API_KEY") or os.getenv("GROQ_API_KEY") else "No")

Environment and packages loaded successfully.
API Key configured: Yes


## 1. Scraping Data Berita
Langkah pertama adalah menarik artikel berita menggunakan RSS feed Google News berdasarkan kata kunci pencarian ritel/FMCG.

In [15]:
from src.scraper.scraper import run_auto_pipeline

# Kueri pencarian retail/FMCG Indonesia
queries = [
    "FMCG Indonesia", 
    "Ritel Modern Indonesia", 
    "Perilaku Konsumen Ritel"
]

print("Menjalankan scraping berita...")
os.makedirs("data_temp", exist_ok=True)

# Run the automated scraper pipeline saving directly to data_temp
raw_path = run_auto_pipeline(queries, limit_per_query=5, output_dir="data_temp", time_range="7d")

with open(raw_path, "r", encoding="utf-8") as f:
    raw_articles = json.load(f)
print(f"Berhasil menarik dan mengunduh {len(raw_articles)} artikel berita.")

2026-07-14 16:57:38,082 - INFO - Loaded 15 existing articles from data_temp/raw_articles.json
2026-07-14 16:57:38,083 - INFO - Searching Google News RSS for query: 'FMCG Indonesia when:7d' (lang: id, country: ID)...


Menjalankan scraping berita...


2026-07-14 16:57:38,496 - INFO - Found 5 article links for query: 'FMCG Indonesia'.
2026-07-14 16:57:39,690 - INFO - Decoded Google News URL to: https://www.bloombergtechnoz.com/detail-news/114473/kenaikan-harga-ubah-pola-belanja-fmcg-murah-jadi-buruan
2026-07-14 16:57:39,691 - INFO - Skipping already scraped URL: https://www.bloombergtechnoz.com/detail-news/114473/kenaikan-harga-ubah-pola-belanja-fmcg-murah-jadi-buruan
2026-07-14 16:57:40,581 - INFO - Decoded Google News URL to: https://industri.kontan.co.id/news/emiten-fmcg-tetap-optimistis-hadapi-semester-ii-2026-meski-daya-beli-melemah
2026-07-14 16:57:40,582 - INFO - Skipping already scraped URL: https://industri.kontan.co.id/news/emiten-fmcg-tetap-optimistis-hadapi-semester-ii-2026-meski-daya-beli-melemah
2026-07-14 16:57:41,515 - INFO - Decoded Google News URL to: https://www.metrotvnews.com/read/bJECjQXy-finance-dan-sales-cek-lowongan-kerja-pt-djojonegoro-c-1000-lulusan-sarjana
2026-07-14 16:57:41,516 - INFO - Skipping already 

Berhasil menarik dan mengunduh 15 artikel berita.


## 2. Preprocessing Data Berita
Pembersihan teks mentah (raw text) dari HTML tag, stopword bahasa Indonesia, dan melakukan tokenisasi serta pembersihan karakter non-alfabetis.

In [16]:
from src.preprocessing.preprocess import run_preprocessing_pipeline

preprocessed_path = "data_temp/preprocessed_articles.json"
print("Menjalankan preprocessing teks...")
run_preprocessing_pipeline(input_path=raw_path, output_path=preprocessed_path)

with open(preprocessed_path, "r", encoding="utf-8") as f:
    prep_data = json.load(f)
print(f"Berhasil memproses {len(prep_data)} artikel.")

2026-07-14 16:58:05,840 - INFO - Loaded 15831 stemmed words from cache file.
2026-07-14 16:58:05,841 - INFO - Loading raw dataset from: data_temp/raw_articles.json
2026-07-14 16:58:05,848 - INFO - Preprocessing 15 articles...
2026-07-14 16:58:05,886 - INFO - Processed 15/15 articles. Stem cache size: 15831
2026-07-14 16:58:05,907 - INFO - Preprocessing complete. Cleaned dataset saved to: data_temp/preprocessed_articles.json
2026-07-14 16:58:05,958 - INFO - Saved 15831 stemmed words to cache file.


Menjalankan preprocessing teks...
Berhasil memproses 15 artikel.


## 3. Topic Modeling dan Clustering
Mengelompokkan artikel berita ke dalam klaster berdasarkan TF-IDF dan K-Means, serta mencocokkan kata kunci dominan ke 10 kategori tren utama.

In [17]:
from src.modeling.topic_model import run_clustering_pipeline

clustered_path = "data_temp/clustered_articles.json"
print("Menjalankan Topic Modeling & Clustering...")
# Jalankan clustering ke 3 klaster dominan
run_clustering_pipeline(input_path=preprocessed_path, output_path=clustered_path, summary_path="data_temp/clustering_summary.json")

with open(clustered_path, "r", encoding="utf-8") as f:
    clustered_data = json.load(f)
print("Artikel berhasil dikelompokkan ke dalam klaster.")

2026-07-14 16:58:10,402 - INFO - Loading preprocessed dataset from: data_temp/preprocessed_articles.json
2026-07-14 16:58:10,411 - INFO - Vectorizing 15 documents...
2026-07-14 16:58:10,425 - INFO - Clustering into 3 clusters...
2026-07-14 16:58:10,479 - INFO - Similarity Score Matrix (Clusters vs Categories):
[[0.08445829 0.15430678 0.5955243  0.02579426 0.03534821 0.02579426
  0.05865377 0.         0.         0.02502482]
 [0.00732177 0.05606515 0.67742133 0.02818539 1.13011875 0.07083068
  0.45534405 0.         0.00732177 0.55118926]
 [0.11027093 0.16759907 1.22035614 0.10822476 0.02878274 0.2337339
  0.20588973 0.         0.02849445 0.01209755]]
2026-07-14 16:58:10,480 - INFO - Mapped Cluster 2 to Category 'Consumer Behavior Shift' (Score: 1.2204)
2026-07-14 16:58:10,481 - INFO - Mapped Cluster 1 to Category 'SME Digitalization' (Score: 1.1301)
2026-07-14 16:58:10,482 - INFO - Mapped Cluster 0 to Category 'Digital Marketing' (Score: 0.1543)
2026-07-14 16:58:10,483 - INFO - Top keywo

Menjalankan Topic Modeling & Clustering...


2026-07-14 16:58:10,611 - INFO - Clustered dataset successfully saved to: data_temp/clustered_articles.json
2026-07-14 16:58:10,617 - INFO - Clustering summary metadata saved to: data_temp/clustering_summary.json


Artikel berhasil dikelompokkan ke dalam klaster.


## 4. AI Summarization (Extractive & Generative)
Menghasilkan ringkasan berita secara Extractive (menggunakan algoritma TextRank lokal) dan Generative (menggunakan model Groq API secara online).

In [18]:
from src.summarizer.summary_engine import run_summarization_pipeline

summary_report_path = "data_temp/final_summary_report.json"

print("Menghasilkan ringkasan industri (Extractive & Generative via Groq API)...")
# Gunakan model API 'groq'
run_summarization_pipeline(
    input_path=clustered_path,
    output_path=summary_report_path,
    model_source="groq"
)

with open(summary_report_path, "r", encoding="utf-8") as f:
    summary_data = json.load(f)
print(f"Ringkasan berhasil dihasilkan untuk {len(summary_data)} kategori tren aktif.\n")

for category, data in summary_data.items():
    print("==================================================")
    print(f"KATEGORI TREN: {category}")
    print(f"Keywords: {', '.join(data['top_keywords'])}")
    print("==================================================")
    
    print("\n[ Hasil Ringkasan Extractive (TextRank) ]")
    print("-" * 50)
    print(data['extractive_brief'])
    print("-" * 50)
    
    print("\n[ Hasil Ringkasan Generative (Groq API) ]")
    print("-" * 50)
    print(data['generative_brief'])
    print("-" * 50)
    print("\n")

2026-07-14 16:58:13,979 - INFO - Loading clustered articles from: data_temp/clustered_articles.json
2026-07-14 16:58:13,987 - INFO - Processing category 'Consumer Behavior Shift' containing 7 articles...
2026-07-14 16:58:14,043 - INFO - Generating generative summary and insights using model: test-1 at base: https://adnandi-9router.hf.space/v1...


Menghasilkan ringkasan industri (Extractive & Generative via Groq API)...


2026-07-14 16:58:36,068 - INFO - Generative summary and insights generated using Groq API successfully.
2026-07-14 16:58:36,069 - INFO - Processing category 'Digital Marketing' containing 3 articles...
2026-07-14 16:58:36,100 - INFO - Generating generative summary and insights using model: test-1 at base: https://adnandi-9router.hf.space/v1...
2026-07-14 16:58:55,781 - INFO - Generative summary and insights generated using Groq API successfully.
2026-07-14 16:58:55,783 - INFO - Processing category 'SME Digitalization' containing 5 articles...
2026-07-14 16:58:55,889 - INFO - Generating generative summary and insights using model: test-1 at base: https://adnandi-9router.hf.space/v1...
2026-07-14 16:59:11,285 - INFO - Generative summary and insights generated using Groq API successfully.
2026-07-14 16:59:11,292 - INFO - Final summary report successfully saved to: data_temp/final_summary_report.json


Ringkasan berhasil dihasilkan untuk 3 kategori tren aktif.

KATEGORI TREN: Consumer Behavior Shift
Keywords: harga, konsumen, beli, ritel, mal, perilaku, jual, turun

[ Hasil Ringkasan Extractive (TextRank) ]
--------------------------------------------------
Bloomberg Technoz, Jakarta - Pelaku usaha ritel di segmen fast moving consumer goods (FMCG) atau produk ritel utama mengakui kenaikan harga sejumlah produk mendorong konsumen menjadi lebih selektif dan mulai beralih ke merek dengan harga yang lebih terjangka selain faktor musiman. "Pada umumnya tekanan daya beli ini membuat konsumen lebih sensitif terhadap harga dan dapat mengubah preferensi konsumen, mereka condong untuk membeli produk-produk single pack dengan harga yang lebih terjangkau," ujar manajemen Mayora kepada Kontan.co.id, Senin (6/7/2026). Di tengah dinamika perilaku konsumen tersebut, Ferry berpendapat bahwa kunci sukses ruang ritel atau mal terletak pada bauran penyewa, pengalaman belanja, dan jumlah pengunjung. Penj

## 5. Model Evaluation (ROUGE Score)
Langkah terakhir adalah mengevaluasi kualitas ringkasan yang dihasilkan oleh model AI (Extractive vs Generative) terhadap Ground Truth buatan manusia.

In [19]:
from src.evaluation.eval_summary import evaluate_summaries

evaluation_results_path = "data_temp/evaluation_results.json"
ground_truth_path = "evaluation/ground_truth.json"

print("Menjalankan ROUGE Evaluation...")
evaluate_summaries(
    report_path=summary_report_path,
    ground_truth_path=ground_truth_path,
    output_path=evaluation_results_path
)

with open(evaluation_results_path, "r", encoding="utf-8") as f:
    eval_results = json.load(f)
print("Evaluasi selesai dan metrik ROUGE tersimpan.")

2026-07-14 17:00:55,574 - INFO - Initializing ROUGE evaluation...
2026-07-14 17:00:55,584 - INFO - Using default tokenizer.
2026-07-14 17:00:55,585 - WARNING - Category 'Sustainability' from ground truth not found in summary report.
2026-07-14 17:00:55,618 - WARNING - Category 'Fintech & Payment Gateway' from ground truth not found in summary report.
2026-07-14 17:00:55,638 - WARNING - Category 'E-commerce & Social Commerce' from ground truth not found in summary report.
2026-07-14 17:00:55,639 - WARNING - Category 'Modern Retail & Supply Chain' from ground truth not found in summary report.
2026-07-14 17:00:55,639 - WARNING - Category 'F&B Industry & Culinary Trends' from ground truth not found in summary report.
2026-07-14 17:00:55,640 - WARNING - Category 'Circular Economy & Plastic Waste' from ground truth not found in summary report.
2026-07-14 17:00:55,641 - WARNING - Category 'Local Brand Empowerment' from ground truth not found in summary report.
2026-07-14 17:00:55,651 - INFO 

Menjalankan ROUGE Evaluation...

TREND CATEGORY            | METRIC   | TYPE       | PRECISION | RECALL | F-MEASURE
Digital Marketing         | rouge1   | Extractive | 0.1966    | 0.2110 | 0.2035   
Digital Marketing         | rouge1   | Generative | 0.3627    | 0.3394 | 0.3507   
--------------------------------------------------------------------------------
Digital Marketing         | rouge2   | Extractive | 0.0259    | 0.0278 | 0.0268   
Digital Marketing         | rouge2   | Generative | 0.0990    | 0.0926 | 0.0957   
--------------------------------------------------------------------------------
Digital Marketing         | rougeL   | Extractive | 0.0940    | 0.1009 | 0.0973   
Digital Marketing         | rougeL   | Generative | 0.2255    | 0.2110 | 0.2180   
--------------------------------------------------------------------------------
Consumer Behavior Shift   | rouge1   | Extractive | 0.1571    | 0.2136 | 0.1811   
Consumer Behavior Shift   | rouge1   | Generative | 0.2130  

## 6. Visualisasi Hasil Evaluasi ROUGE
Mari kita tampilkan skor ROUGE-1, ROUGE-2, dan ROUGE-L secara visual untuk membandingkan performa model Extractive vs Generative.

In [20]:
# Tampilkan tabel skor ROUGE
rows = []
for category, models in eval_results.items():
    for model_type in ["extractive", "generative"]:
        metrics = models[model_type]
        rows.append({
            "Kategori": category,
            "Tipe Model": model_type.capitalize(),
            "ROUGE-1 F1": f"{metrics['rouge1']['fmeasure']*100:.2f}%",
            "ROUGE-2 F1": f"{metrics['rouge2']['fmeasure']*100:.2f}%",
            "ROUGE-L F1": f"{metrics['rougeL']['fmeasure']*100:.2f}%"
        })

df_eval = pd.DataFrame(rows)
print("=== Tabel Perbandingan Hasil Evaluasi ===")
print(df_eval.to_string(index=False))

=== Tabel Perbandingan Hasil Evaluasi ===
               Kategori Tipe Model ROUGE-1 F1 ROUGE-2 F1 ROUGE-L F1
      Digital Marketing Extractive     20.35%      2.68%      9.73%
      Digital Marketing Generative     35.07%      9.57%     21.80%
Consumer Behavior Shift Extractive     18.11%      0.00%      9.88%
Consumer Behavior Shift Generative     26.47%      8.15%     16.91%
     SME Digitalization Extractive     26.67%      2.52%     14.17%
     SME Digitalization Generative     32.28%     10.11%     18.52%
